# Chapter 5 — Programs are values

Step 4 builds the IR — and with it, the system's deepest structural claim:
**the anti-pattern is unrepresentable.** A `Node` has no field that can hold
a runtime value; `attrs` is the single, deliberate carve-out for compile-time
constants, inside identity and visible in the printed text. A captured value
is a slot number here, never a value.

| File | Counted lines / cap | What |
|---|---|---|
| `src/pdum/dsl/kernel/ir.py` | 84 / 150 | `Node`, `Region`, `Loc`, content key, `Builder`, `verify` |
| `src/pdum/dsl/kernel/ops.py` | 88 / 110 | `OpDef` + the ~22-op core dialect table (a dialect IS a dict) |
| `src/pdum/dsl/kernel/printer.py` | 45 / 80 | the MLIR-flavored textual form (golden tests; migration insurance) |

Glossary terms settled: *Node/Region, content key, dialect (as a dict of
OpDefs).*

In [1]:
from pdum.dsl.kernel import types as T
from pdum.dsl.kernel.ir import Builder, Loc, Node, Region
from pdum.dsl.kernel.ops import CORE_OPS
from pdum.dsl.kernel.printer import print_program

b = Builder(CORE_OPS)


def disk_body():
    """The disk shader's decision, built by hand: (x-cx)^2 + (y-cy)^2 < r^2."""
    x, y = b.param(0, T.f64), b.param(1, T.f64)
    cx = b.emit("core.env", type=T.f64, slot=0)
    cy = b.emit("core.env", type=T.f64, slot=1)
    r = b.emit("core.env", type=T.f64, slot=2)
    dx, dy = b.emit("core.sub", x, cx), b.emit("core.sub", y, cy)
    d2 = b.emit("core.add", b.emit("core.mul", dx, dx), b.emit("core.mul", dy, dy))
    hit = b.emit("core.cmp", d2, b.emit("core.mul", r, r), pred="lt")
    return Region(params=(x, y), body=(b.emit("core.yield", hit),))


prog = disk_body()
print(print_program(prog, name="disk"))

disk(%p0: f64, %p1: f64) {
  %0 = core.env {slot = 0} : f64
  %1 = core.sub %p0, %0 : f64
  %2 = core.mul %1, %1 : f64
  %3 = core.env {slot = 1} : f64
  %4 = core.sub %p1, %3 : f64
  %5 = core.mul %4, %4 : f64
  %6 = core.add %2, %5 : f64
  %7 = core.env {slot = 2} : f64
  %8 = core.mul %7, %7 : f64
  %9 = core.cmp %6, %8 {pred = 'lt'} : bool
  core.yield %9
}


In [2]:
from pdum.dsl import viz

viz.install()
prog  # the same program, rendered: hover any %ref for its type and provenance

Region(params=(Node(op='core.param', type=f64, args=(), regions=(), attrs=(('index', 0),), loc=None), Node(op='core.param', type=f64, args=(), regions=(), attrs=(('index', 1),), loc=None)), body=(Node(op='core.yield', type=bool, args=(Node(op='core.cmp', type=bool, args=(Node(op='core.add', type=f64, args=(Node(op='core.mul', type=f64, args=(Node(op='core.sub', type=f64, args=(Node(op='core.param', type=f64, args=(), regions=(), attrs=(('index', 0),), loc=None), Node(op='core.env', type=f64, args=(), regions=(), attrs=(('slot', 0),), loc=None)), regions=(), attrs=(), loc=None), Node(op='core.sub', type=f64, args=(Node(op='core.param', type=f64, args=(), regions=(), attrs=(('index', 0),), loc=None), Node(op='core.env', type=f64, args=(), regions=(), attrs=(('slot', 0),), loc=None)), regions=(), attrs=(), loc=None)), regions=(), attrs=(), loc=None), Node(op='core.mul', type=f64, args=(Node(op='core.sub', type=f64, args=(Node(op='core.param', type=f64, args=(), regions=(), attrs=(('index', 1),), loc=None), Node(op='core.env', type=f64, args=(), regions=(), attrs=(('slot', 1),), loc=None)), regions=(), attrs=(), loc=None), Node(op='core.sub', type=f64, args=(Node(op='core.param', type=f64, args=(), regions=(), attrs=(('index', 1),), loc=None), Node(op='core.env', type=f64, args=(), regions=(), attrs=(('slot', 1),), loc=None)), regions=(), attrs=(), loc=None)), regions=(), attrs=(), loc=None)), regions=(), attrs=(), loc=None), Node(op='core.mul', type=f64, args=(Node(op='core.env', type=f64, args=(), regions=(), attrs=(('slot', 2),), loc=None), Node(op='core.env', type=f64, args=(), regions=(), attrs=(('slot', 2),), loc=None)), regions=(), attrs=(), loc=None)), regions=(), attrs=(('pred', 'lt'),), loc=None),), regions=(), attrs=(), loc=None),))

Read the text like MLIR: every node is an SSA value with exactly one result
type; captures appear as `core.env {slot = k}` — a *slot*, never a value;
types are the honest widths (`f64` — narrowing to `f32` is a backend
`type_map` decision at render, chapters away).

In [3]:
# Content identity: a memoized sha256 over structure — the artifact-tier key.
print("key:              ", prog.key.hex()[:16], "…")
print("rebuilt:          ", disk_body().key == prog.key)

b2 = Builder(CORE_OPS)
x, y = b2.param(0, T.f64), b2.param(1, T.f64)
tweaked = Region((x, y), (b2.emit("core.yield", b2.emit("core.env", type=T.f64, slot=7)),))
print("tweaked shares key?", tweaked.key == prog.key, " <- different structure, different key")

# `loc` — where code came from — is EXCLUDED from identity:
n1 = b.emit("core.add", x, y, loc=Loc("notebook.py", 1))
n2 = b.emit("core.add", x, y, loc=Loc("elsewhere.py", 99))
print("loc excluded:     ", n1 == n2 and n1.key == n2.key)

key:               8fa7c01c8275fab9 …
rebuilt:           True
tweaked shares key? False  <- different structure, different key
loc excluded:      True


In [4]:
import dataclasses

# THE invariant, inspected: no field on Node can hold a runtime value.
for f in dataclasses.fields(Node):
    if not f.name.startswith("_"):
        print(f"  Node.{f.name:<8} : {f.type}")

try:
    Node(op="core.env", type=T.f64, value=5)  # there is no such field
except TypeError as e:
    print("smuggling a value ->", type(e).__name__, "-", e)

  Node.op       : str
  Node.type     : Type
  Node.args     : tuple[Node, ...]
  Node.regions  : tuple[Region, ...]
  Node.attrs    : tuple[Attr, ...]
  Node.loc      : Provenance | None
smuggling a value -> TypeError - Node.__init__() got an unexpected keyword argument 'value'


In [5]:
# The one value-shaped slot, used deliberately — and visibly:
env = b.emit("core.env", type=T.f64, slot=0)  # runtime capture: re-marshaled per call
const = b.emit("core.const", type=T.f64, value=8.0)  # Literal lift: value IN identity
print(env.attrs, " vs ", const.attrs)
print(
    "their keys differ per value?",
    b.emit("core.const", type=T.f64, value=9.0).key != const.key,
    " <- const recompiles per value; env never does. That difference IS the thesis.",
)

(('slot', 0),)  vs  (('value', 8.0),)
their keys differ per value? True  <- const recompiles per value; env never does. That difference IS the thesis.


In [6]:
# Structural sharing is real (a DAG, not a tree) — and the printer shows it:
e = b.emit("core.env", type=T.f64, slot=0)
sq = b.emit("core.mul", e, e)
total = b.emit("core.add", sq, sq)  # the SAME node object, used twice
print(print_program(Region(body=(b.emit("core.yield", total),)), name="shared"))

shared() {
  %0 = core.env {slot = 0} : f64
  %1 = core.mul %0, %0 : f64
  %2 = core.add %1, %1 : f64
  core.yield %2
}


In [7]:
# Structured control flow: exactly three region ops (if / for / call), forever
# until forced — a fourth is priced at ~180 lines x live transform columns.
f = b.emit("core.env", type=T.f64, slot=0)
cond = b.emit("core.cmp", f, b.emit("core.const", type=T.f64, value=0.5), pred="lt")
then = Region(body=(b.emit("core.yield", f),))
other = Region(body=(b.emit("core.yield", b.emit("core.neg", f)),))
branch = b.emit("core.if", cond, regions=(then, other))
print(print_program(Region(body=(b.emit("core.yield", branch),)), name="abs_ish"))

try:  # branches must agree — checked at CONSTRUCTION, not at render
    i = b.emit("core.env", type=T.i64, slot=1)
    b.emit("core.if", cond, regions=(then, Region(body=(b.emit("core.yield", i),))))
except TypeError as e:
    print("\nrefused:", e)

abs_ish() {
  %0 = core.env {slot = 0} : f64
  %1 = core.const {value = 0.5} : f64
  %2 = core.cmp %0, %1 {pred = 'lt'} : bool
  %4 = core.if %2 ({
    core.yield %0
  }, {
    %3 = core.neg %0 : f64
    core.yield %3
  }) : f64
  core.yield %4
}

refused: core.if branches yield f64 vs i64


In [8]:
# Core arithmetic is STRICT (settled at this chapter's walkthrough): operands
# must share a type — there is NO promotion in the kernel. Promotion, where a
# language wants it, is a DIALECT's lowering policy (auto-insert casts), or
# absent (write float(i) yourself — the Julia model: cast, then same-type add).
from pdum.dsl.kernel.ir import VerifyError

i, fl = b.emit("core.env", type=T.i64, slot=0), b.emit("core.env", type=T.f64, slot=1)
print("i64 + i64 ->", b.emit("core.add", i, i).type)
try:
    b.emit("core.add", i, fl)
except TypeError as e:
    print("i64 + f64 ->", e)
w = b.emit("core.cast", i, to=T.f64)
print("cast(i) + f64 ->", b.emit("core.add", w, fl).type, "  <- the cast is IN the IR, in the hash")

v = b.emit("core.vec", fl, fl, fl)
print(v.type, "->", b.emit("core.extract", v, index=2).type, "|", b.emit("core.cast", fl, to=T.f32).type)

for bad in (lambda: b.emit("core.frobnicate", i), lambda: b.emit("core.env", slot=0)):
    try:
        bad()
    except VerifyError as e:
        print("VerifyError:", e)

i64 + i64 -> i64
i64 + f64 -> core arithmetic is strict: i64 vs f64 — insert an explicit core.cast
cast(i) + f64 -> f64   <- the cast is IN the IR, in the hash
vec3<f64> -> f64 | f32
VerifyError: unknown op 'core.frobnicate'; register it (defop) before emitting
VerifyError: core.env has no type rule; pass an explicit type=


## What we can't do yet

- Nothing *produces* IR but our hands: rewriting arrives in ch06, lowering
  (source → IR, the combinator build rule) in ch07. Every program in this
  chapter was built with the `Builder`, on purpose — the IR is a public,
  inspectable value, not compiler-internal state.
- Nothing consumes it either: legalization (ch08) and rendering (ch09) are
  where `core.env` becomes physical slots and text.
- The artifact tier can now key on `Region.key` for real — ch03's fake
  `"sha256:9f3a..."` strings retire when lowering lands.

**Gates now armed:** the anti-pattern field check and the printer golden test
(`tests/test_ir.py`).

**Budget after step 4:** kernel 595 / 1150 counted lines.

**Next:** ch06 — everything is a rule: `Pat`/`RuleSet`, the one rewrite
driver, and stage legality.